# 04 — Bronze: micro-lotes em Delta com metadados

**Squad 2 — Negócio em Tempo Real | Dupla José Vieira & Tiago Ramos**
**Tabela:** `ecommerce_itens_pedido` · **Destino:** `TABELA_BRONZE` (Delta, Unity Catalog)

A cada 5 segundos verifica se o gerador gravou um arquivo novo de `ecommerce_itens_pedido` no Data Lake.
Arquivo novo é lido com Spark e gravado na Bronze **sem transformação nos dados**, só com as colunas de auditoria.

| # | Regra | Como |
|---|---|---|
| B1 | Só entra arquivo que ainda não está na Bronze | comparação por `bronze_source_file` |
| B2 | Nenhuma transformação nas colunas de origem | leitura com `SCHEMA_ORIGEM`, sem cast nem filtro |
| B3 | `bronze_ingested_at` = momento da leitura | `current_timestamp()` |
| B4 | `bronze_source_file` = arquivo de origem | caminho relativo ao container, sem o nome da conta |
| B5 | Partição pela data de ingestão | `ano`, `mes`, `dia`, `hora` (horário de Brasília); a tabela não tem data de negócio |
| B6 | Gravação só acrescenta | Delta em modo `append` |
| B7 | O loop sempre termina | teto de duração e parada por ociosidade |

In [0]:
%run ./00_setup_config

## Parâmetros

O loop para pelo que acontecer primeiro: `DURACAO_MAX_MIN` de execução ou `OCIOSO_MAX_MIN` sem arquivo
novo (sinal de que a janela fechou). Com o widget `loop_ativo = false` faz uma passada só: é o modo do
Job agendado e o de recuperar lotes antigos.

In [0]:
import time
from datetime import datetime
from zoneinfo import ZoneInfo
from pyspark.sql import functions as F

INTERVALO_S, DURACAO_MAX_MIN, OCIOSO_MAX_MIN = 5, 30, 10
dbutils.widgets.text("loop_ativo", "true")
LOOP_ATIVO = dbutils.widgets.get("loop_ativo").strip().lower() == "true"
COLUNAS_PARTICAO = ["ano", "mes", "dia", "hora"]
RAIZ_CONTAINER = f"abfss://{CONTAINER}@{_sufixo}/"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SQUAD}")
print(f"Bronze: {TABELA_BRONZE} | modo: {'loop a cada %ss, max %s min, para apos %s min ocioso' % (INTERVALO_S, DURACAO_MAX_MIN, OCIOSO_MAX_MIN) if LOOP_ATIVO else 'uma passada'}")

## 1. Como o arquivo novo é detectado

As credenciais por chamada não funcionam em `dbutils.fs.ls`, então a listagem também é uma leitura Spark:
o formato `binaryFile` com o mesmo glob do `01` devolve uma linha por arquivo, e só a coluna `path` é
selecionada (o conteúdo não é lido). O que já foi ingerido vem da própria Bronze. A diferença são os
arquivos novos; esses conjuntos são nomes de arquivo, por isso podem ficar no driver.

In [0]:
relativo = lambda c: F.regexp_extract(c, rf"({PREFIXO}/.+)$", 1)


def listar_arquivos():
    try:
        linhas = spark.read.format("binaryFile").options(**adls_options).load(CAMINHO_ORIGEM).select(relativo(F.col("path")).alias("a")).collect()
    except Exception as e:
        msg = str(e)
        if "PATH_NOT_FOUND" in msg or "does not exist" in msg:
            return set()
        if "AuthorizationPermissionMismatch" in msg or "403" in msg:
            raise PermissionError("403: o service principal autenticou, mas nao tem leitura no container.") from None
        if "AADSTS" in msg or "invalid_client" in msg or "401" in msg:
            raise PermissionError("401: credencial recusada - conferir o .env.") from None
        raise
    return {r["a"] for r in linhas}


def ja_ingeridos():
    if not spark.catalog.tableExists(TABELA_BRONZE):
        return set()
    return {r[0] for r in spark.table(TABELA_BRONZE).select("bronze_source_file").distinct().collect()}


no_lake, na_bronze = listar_arquivos(), ja_ingeridos()
print(f"No lake: {len(no_lake)} | ja na Bronze: {len(na_bronze)} | a ingerir: {len(no_lake - na_bronze)}")

## 2. Leitura e colunas de auditoria

Ler sempre com `SCHEMA_ORIGEM` protege a tabela: o tipo não muda de um lote para outro, e uma coluna nova
do gerador não entra sozinha. Uma coluna que **falte** no arquivo chega como nula em todas as linhas dele;
a Silver detecta isso (regra T1).

In [0]:
def ingerir(arquivos):
    local = F.expr(f"convert_timezone('{FUSO_NEGOCIO}', bronze_ingested_at)")
    df = (spark.read.options(**adls_options).schema(SCHEMA_ORIGEM).parquet(*[RAIZ_CONTAINER + a for a in sorted(arquivos)])
          .withColumn("bronze_ingested_at", F.current_timestamp()).withColumn("bronze_source_file", relativo(F.col("_metadata.file_path")))
          .withColumn("ano", F.year(local)).withColumn("mes", F.month(local)).withColumn("dia", F.dayofmonth(local)).withColumn("hora", F.hour(local)))
    df.write.format("delta").mode("append").partitionBy(*COLUNAS_PARTICAO).saveAsTable(TABELA_BRONZE)
    return spark.table(TABELA_BRONZE).filter(F.col("bronze_source_file").isin(list(arquivos))).count()  # confere o que ficou gravado

## 3. Loop de micro-lotes

A cada ciclo: lista o lake, separa o que é novo, ingere e espera `INTERVALO_S` segundos. Rodar de novo
não duplica nada (regra B1). O botão **Interromper** encerra o loop normalmente e mostra o resumo.

In [0]:
agora = lambda: datetime.now(ZoneInfo(FUSO_NEGOCIO)).strftime("%H:%M:%S")
ingeridos, inicio = ja_ingeridos(), time.monotonic()
ultimo_novo, ciclos, arquivos_gravados, linhas_gravadas, motivo = inicio, 0, 0, 0, "passada unica concluida"
print(f"[{agora()}] inicio - {len(ingeridos)} arquivo(s) ja na Bronze")
try:
    while True:
        ciclos += 1
        novos = listar_arquivos() - ingeridos
        if novos:
            linhas = ingerir(novos)
            ingeridos |= novos; arquivos_gravados += len(novos); linhas_gravadas += linhas; ultimo_novo = time.monotonic()
            print(f"[{agora()}] +{len(novos)} arquivo(s), {linhas} linha(s)")
        if not LOOP_ATIVO:
            break
        if time.monotonic() - inicio >= DURACAO_MAX_MIN * 60:
            motivo = f"teto de {DURACAO_MAX_MIN} min atingido"; break
        if time.monotonic() - ultimo_novo >= OCIOSO_MAX_MIN * 60:
            motivo = f"{OCIOSO_MAX_MIN} min sem arquivo novo - janela provavelmente fechada"; break
        time.sleep(INTERVALO_S)
except KeyboardInterrupt:
    motivo = "interrompido manualmente"
print(f"[{agora()}] fim: {motivo} | ciclos {ciclos} | arquivos {arquivos_gravados} | linhas {linhas_gravadas}")

## 4. Validação da Bronze

| Verificação | Esperado |
|---|---|
| Arquivos do lake que ainda não estão na Bronze | 0 |
| Arquivos ingeridos mais de uma vez | 0 |
| Linhas sem `bronze_source_file` | 0 |

`id_item_pedido` repetido é **informativo**: a Bronze guarda o que chegou, e deduplicar é papel da Silver (T5).

In [0]:
if not spark.catalog.tableExists(TABELA_BRONZE):
    print(f"[aviso] {TABELA_BRONZE} ainda nao existe: nenhum arquivo foi ingerido.")
else:
    bronze = spark.table(TABELA_BRONZE)
    por_arquivo = bronze.groupBy("bronze_source_file").agg(F.count("*").alias("linhas"), F.countDistinct("bronze_ingested_at").alias("ingestoes"))
    r = bronze.agg(F.count("*").alias("linhas"), F.countDistinct("bronze_source_file").alias("arquivos"), F.countDistinct(CHAVE_PRIMARIA).alias("chaves"),
                   F.sum((F.coalesce("bronze_source_file", F.lit("")) == "").cast("int")).alias("sem_origem")).first()
    pendentes = listar_arquivos() - {x[0] for x in por_arquivo.select("bronze_source_file").collect()}
    print(f"Linhas: {r['linhas']} | arquivos: {r['arquivos']} | {CHAVE_PRIMARIA} repetidos: {r['linhas'] - r['chaves']} (informativo)")
    for nome, valor in [("arquivos do lake fora da Bronze", len(pendentes)), ("arquivos ingeridos mais de uma vez", por_arquivo.filter("ingestoes > 1").count()),
                        ("linhas sem bronze_source_file", r["sem_origem"] or 0)]:
        print(f"[{'ok' if valor == 0 else '!!'}] {nome}: {valor}")
    display(bronze.groupBy(*COLUNAS_PARTICAO).count().orderBy(*COLUNAS_PARTICAO))